# Smart Lender – EDA & Model Training


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib


## 1. Load Data


In [ ]:
df = pd.read_csv('../data/archive/train_u6lujuX_CVtuZ9i.csv')
df.head()


## 2. EDA


In [ ]:
print(df.shape)
print(df.isnull().sum())
df.describe()


In [ ]:
sns.countplot(x='Loan_Status', data=df)
plt.title('Loan Status Distribution')
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
cat_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Credit_History', 'Property_Area']
for ax, col in zip(axes.flatten(), cat_cols):
    sns.countplot(x=col, hue='Loan_Status', data=df, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


## 3. Preprocessing


In [ ]:
df.drop(columns=['Loan_ID'], inplace=True, errors='ignore')

for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)
for col in df.select_dtypes(include='number').columns:
    df[col].fillna(df[col].median(), inplace=True)

le = LabelEncoder()
if df['Loan_Status'].dtype == object:
    df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 4. Train & Evaluate All Models


In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc  = accuracy_score(y_test,  model.predict(X_test))
    results[name] = (train_acc, test_acc)
    print(f'{name}: Train={train_acc:.4f}, Test={test_acc:.4f}')


In [ ]:
res_df = pd.DataFrame(results, index=['Train Accuracy', 'Test Accuracy']).T
res_df.plot(kind='bar', figsize=(10, 5), ylim=(0.5, 1.0))
plt.title('Model Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Save Best Model (XGBoost)


In [ ]:
best_model = models['XGBoost']
joblib.dump(best_model, '../model/best_model.pkl')
print('Model saved to ../model/best_model.pkl')
